In [35]:
from datasets import load_dataset
import pandas as pd 

REPO = "bakrianoo/jabarti-llm-dataset" 

pretrain = load_dataset(REPO, "pretrain", split="train").to_pandas()  # or split="eval"
finetune = load_dataset(REPO, "finetune", split="train").to_pandas()  # or split="eval"



In [36]:
print("===pretrain schema===")
print(pretrain.dtypes)


===pretrain schema===
article_id              str
title                   str
url                     str
language                str
category                str
source_type             str
egypt_relevance         str
content_quality         str
topic_domain            str
importance              str
is_person            object
is_stub_like         object
ortho_convention        str
pageviews_30d       float64
chunk_strategy          str
chunk_index         float64
chunk_total         float64
pair_id                 str
dataset_version         str
ds_source               str
text                    str
section_index       float64
section_total       float64
ortho_stripped       object
page_id                 str
record_id               str
sample_weight       float64
dtype: object


In [37]:
print("===finetune schema===")
print(finetune.dtypes)


===finetune schema===
difficulty           str
type                 str
qa_id                str
dataset_version      str
question             str
article_title        str
answer               str
source_article_id    str
language             str
dtype: object


In [ ]:
splits = {"pretrain / phase1_train" : ("pretrain" ,"phase1_train") , 
          "pretrain / phase1_eval" : ("pretrain" ,"phase1_eval") ,
          "pretrain / phase2_train" : ("pretrain" ,"phase2_train"), 
          "pretrain / phase2_eval" : ("pretrain" ,"phase2_eval"),

          "finetune / train" : ("finetune" ,"train"),
          "finetune / eval" : ("finetune" ,"eval")  
}
CHARS_PER_TOKEN = 3.5
CONTEXT_LENGTHS = [512, 1024, 2048] 
for label_name, (subset , split_name) in splits.items() : 
    df = load_dataset(REPO, subset, split=split_name).to_pandas() 

    print(f"\n ====== {label_name}")
    print("ROWS: ",df.shape[0])

    nulls = df.isnull().sum()
    non_nulls = nulls[ nulls > 0 ]

    print(
    "NONE"
    if non_nulls.empty
    else non_nulls.to_string()

    )
    # Check Distribution 
    total_rows = df.shape[0]
    lang_counts = df["language"].value_counts()
    for lang , n in lang_counts.items():
        print(f"{lang} ==> {n}")
        print(f"{(n/total_rows)*100:.2f}%")

    # Text length distribution
    text_len_dist = 0
    if subset == "pretrain":
        text_len_dist = df["text"].str.len().describe().round(1)

    elif subset == "finetune":
        text_len_dist = pd.concat([
        df["question"].str.len(),
        df["answer"].str.len()
        ], ignore_index=True).describe().round(1)

    print("text_len_dist:", text_len_dist)

    #============================================
    # Approximate token length per MODEL SAMPLE
    #============================================
    if subset == "pretrain" :
        sample_chars = df["text"].str.len() 

    elif subset == "finetune":
        sample_chars = (
            df["question"].str.len() + df["answer"].str.len()

            
        )
    approx_tokens = (sample_chars / CHARS_PER_TOKEN).astype(int)
    print("\n === APPROXIMATE TOKEN COUNT PER SAMPLE === ")
    print(
    approx_tokens
    .describe()
    .round(1))

    
    

    print("============")



 ====== pretrain / phase1_train
ROWS:  175476
NONE
en ==> 125766
71.67%
ar ==> 49710
28.33%
text_len_dist: count    175476.0
mean       1553.0
std         702.1
min         178.0
25%        1169.0
50%        1653.0
75%        1913.0
max       20049.0
Name: text, dtype: float64

 === APPROXIMATE TOKEN COUNT PER SAMPLE === 
count    175476.0
mean        443.3
std         200.6
min          50.0
25%         334.0
50%         472.0
75%         546.0
max        5728.0
Name: text, dtype: float64
current max_length=512|kept: 61.9% |dropped: 38.1%
current max_length=1024|kept: 99.3% |dropped: 0.7%
current max_length=2048|kept: 99.8% |dropped: 0.2%

 ====== pretrain / phase1_eval
ROWS:  2000
NONE
en ==> 1426
71.30%
ar ==> 574
28.70%
text_len_dist: count    2000.0
mean     1565.1
std       577.7
min       268.0
25%      1226.8
50%      1666.5
75%      1928.0
max      7520.0
Name: text, dtype: float64

 === APPROXIMATE TOKEN COUNT PER SAMPLE === 
count    2000.0
mean      446.7
std       165.1
m

In [39]:
df = load_dataset(REPO, "pretrain", split="phase1_train").to_pandas() 
text_dups = df.duplicated("text").sum()
pair_dups = df.duplicated(["article_id", "chunk_index"]).sum()
print(text_dups)
print(pair_dups) 

0
0
